# Chain-of-Thought and Reasoning Prompts

Some tasks are not solved by picking a label from a short instruction — they require **intermediate reasoning**: arithmetic steps, logical deductions, planning, or combining multiple facts. **Chain-of-thought (CoT)** prompting asks the model to show its reasoning **before** the final answer, which often improves accuracy on math, logic, and multi-step problems.

CoT can be triggered with a simple phrase ("think step by step"), with **worked examples** that include reasoning traces, or by using **reasoning models** trained to emit internal reasoning tokens. This notebook explains each approach, when CoT helps or hurts, self-consistency at a high level, and how prompted CoT compares to dedicated reasoning endpoints.

Prerequisites: **01–03** in this section (especially few-shot examples). Inference parameters (`temperature`, `max_tokens`) from **02. GenAI & LLM Fundamentals** affect CoT behavior significantly.

---

## 1. What Is Chain-of-Thought?

**Chain-of-thought prompting** means eliciting a sequence of intermediate reasoning steps that lead to a final answer — not just the answer itself.

**Without CoT (direct answer):**

```
Q: A store has 17 apples. It sells 9 and receives a shipment of 25. How many apples now?
A: 33
```

**With CoT:**

```
Q: A store has 17 apples...
A: Start with 17. After selling 9: 17 - 9 = 8. After shipment: 8 + 25 = 33. Final answer: 33.
```

### 1.1 Why CoT Often Works

Autoregressive models generate one token at a time. If the first tokens must be the **final number**, the model must compress the entire calculation into a single jump. If the first tokens can be **intermediate steps**, each step is a smaller prediction — errors are easier to recover from and the model can "think aloud."

Research on CoT (Wei et al., 2022) showed large gains on math and symbolic reasoning when examples included reasoning chains. The technique transfers to instruction-tuned chat models with short triggers like **"Let's think step by step."**

### 1.2 Zero-Shot CoT vs Few-Shot CoT

| Variant | What you add | Token cost |
|---------|--------------|------------|
| **Zero-shot CoT** | Phrase such as "Think step by step" or "Let's work this out step by step" | Low |
| **Few-shot CoT** | 1–3 full worked examples with visible reasoning before the answer | Higher |

**Zero-shot CoT** is the fastest experiment: append the trigger to an otherwise normal question.

**Few-shot CoT** is stronger when the task has a non-obvious reasoning style (custom accounting rules, domain-specific logic) — you demonstrate exactly how steps should look.

---

## 2. When CoT Improves Results

### 2.1 Tasks That Benefit

| Task type | Why CoT helps |
|-----------|---------------|
| **Arithmetic word problems** | Decomposes into +/− operations |
| **Multi-hop logic** | Tracks intermediate facts |
| **Planning** | Orders steps before execution |
| **Code debugging** | Localizes failure before fix |
| **Ambiguous comparisons** | Makes criteria explicit |

### 2.2 When CoT Adds Little or Hurts

| Situation | Issue |
|-----------|-------|
| Simple classification | Extra tokens, no accuracy gain |
| Creative writing | Over-rationalization stiffens prose |
| Strict latency budgets | Longer completions cost time and money |
| Low `max_tokens` | Reasoning truncated → wrong final answer |
| User should not see reasoning | CoT exposes internal steps (privacy/UX) |

### 2.3 Engineering Checklist for CoT

1. **Raise `max_tokens`** — reasoning needs room (often 256–1024+ for hard problems)
2. **Use low temperature** for math/logic (0–0.3); higher temperature multiplies reasoning paths
3. **Ask for a clear final line** — e.g. "Final answer: X" for parsing
4. **Verify** — CoT can include plausible but wrong steps (**fluent reasoning errors**)

---

## 3. Prompt Patterns for CoT

### 3.1 Zero-Shot Trigger Phrases

Common triggers (equivalent in spirit; test on your task):

- "Let's think step by step."
- "Reason through this carefully before answering."
- "Show your work, then give the final answer on the last line."

Combine with **output format** from notebook 02:

```
Solve the problem. Show each step.
End with exactly: FINAL_ANSWER: <number>
```

### 3.2 Few-Shot CoT Template

```
Question: {q1}
Reasoning: {step-by-step trace}
Answer: {a1}

Question: {q2}
Reasoning: ...
Answer: ...

Question: {new question}
Reasoning:
```

The model continues the pattern. Keep reasoning style **consistent** across examples (same labels, same level of detail).

### 3.3 Structured CoT (Outline First)

For complex tasks, ask for an **outline** before execution:

1. List known facts
2. State sub-questions
3. Solve each sub-question
4. Synthesize final answer

This is CoT with explicit scaffolding — useful for business analysis and debugging.

---

## 4. Self-Consistency (Overview)

**Self-consistency** improves reliability by sampling **multiple reasoning paths** and taking a **majority vote** on the final answer.

Workflow:

1. Send the same CoT prompt **N times** with `temperature > 0` (e.g. 0.7)
2. Parse the final answer from each completion
3. Return the **most frequent** answer

Intuition: wrong reasoning paths often disagree; correct paths often converge. Cost is **N×** a single call — used when accuracy matters more than cost (exams, financial checks).

This notebook does not run a full self-consistency loop (it requires many API calls). The concept pairs naturally with **09. Prompt Evaluation and Iteration** — treat voting as an evaluation strategy.

---

## 5. Reasoning Models vs Prompted CoT

Frontier providers now ship **reasoning-focused models** (e.g. OpenAI o-series, DeepSeek R1) that spend extra compute on internal reasoning tokens before answering.

| Approach | How reasoning appears | You control via |
|----------|----------------------|-----------------|
| **Prompted CoT** | Visible in completion text | Prompt wording, `max_tokens` |
| **Reasoning models** | Often separate reasoning / summary blocks | Model choice, reasoning effort settings |

**When to use prompted CoT on a standard chat model:**

- You already use `gpt-4o-mini` or similar everywhere
- You want transparent steps in the response
- Problem difficulty fits chat + CoT trigger

**When to consider a reasoning model:**

- Hard math, coding, or planning despite strong CoT prompts
- You accept higher latency and cost per request

Prompted CoT remains essential knowledge: even reasoning models respond to clear task decomposition in the user message.

---

## 6. Demonstration: Direct Answer vs Zero-Shot CoT

Same math word problem — first **without** CoT, then with **"Think step by step."** Replace the placeholder API key before running.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"

PROBLEM = (
    "Roger has 5 tennis balls. He buys 2 cans of tennis balls with 3 balls each. "
    "How many tennis balls does he have now?"
)


def ask(user_prompt: str, max_tokens: int = 150) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a careful math tutor."},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content or ""


direct_prompt = f"{PROBLEM}\n\nGive only the final numeric answer."
cot_prompt = (
    f"{PROBLEM}\n\nThink step by step. Show each calculation. "
    "End with a line: FINAL_ANSWER: <number>"
)

print("=== DIRECT (no CoT) ===")
print(ask(direct_prompt, max_tokens=50))
print("\n" + "=" * 50 + "\n")
print("=== ZERO-SHOT CoT ===")
print(ask(cot_prompt, max_tokens=300))

The correct answer is **11** (5 + 2×3). Compare whether direct mode states 11 without steps, and whether CoT shows `5 + 6 = 11`. Models usually get this problem right; harder variants below stress CoT more.

---

## 7. Demonstration: Harder Problem — Direct vs CoT

A multi-step discount problem where direct answers are wrong more often.

In [ ]:
HARD_PROBLEM = (
    "A jacket costs $80. It is discounted by 25%, then an extra 10% is taken off the "
    "discounted price. What is the final price in dollars?"
)

direct = f"{HARD_PROBLEM}\nAnswer with one number only."
cot = (
    f"{HARD_PROBLEM}\n\nWork step by step. "
    "After each step, state the intermediate price. "
    "Final line: FINAL_ANSWER: <number>"
)

print("Expected: 80 -> 60 after 25% off -> 54 after 10% off\n")
print("=== DIRECT ===")
print(ask(direct, max_tokens=30))
print("\n=== CoT ===")
print(ask(cot, max_tokens=350))

A common mistake is applying both discounts to the original $80 (getting $52) instead of sequentially ($54). CoT makes the sequential structure visible.

---

## 8. Demonstration: Few-Shot CoT

One worked example teaches the reasoning **format** before a new question.

In [ ]:
FEW_SHOT_COT = """Question: Maria had 12 stickers. She gave 4 to a friend and bought a pack of 7. How many stickers now?
Reasoning: Start with 12. After giving away 4: 12 - 4 = 8. After buying 7: 8 + 7 = 15.
Answer: 15

Question: A class has 30 students. 40% are girls. How many girls?
Reasoning: Girls = 40% of 30 = 0.4 * 30 = 12.
Answer: 12

Question: Tom reads 15 pages per day for 4 days, then 10 pages on day 5. Total pages?
Reasoning:"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "Continue the pattern. Finish Reasoning, then Answer on its own line.",
        },
        {"role": "user", "content": FEW_SHOT_COT},
    ],
    temperature=0.0,
    max_tokens=200,
)

print(FEW_SHOT_COT)
print(response.choices[0].message.content)

Expected reasoning: days 1–4 → 15×4 = 60, plus day 5 → 10, total **70**. Few-shot CoT teaches both arithmetic style and output labels (`Reasoning:` / `Answer:`).

---

## 9. Parsing the Final Answer from CoT Text

Applications often need the **final value** programmatically. Ask for a machine-friendly suffix:

```
FINAL_ANSWER: 54
```

Then parse in code (demo below). Structured output notebooks cover JSON schemas; this pattern is a lightweight middle ground.

In [ ]:
import re

cot_text = ask(cot, max_tokens=350)

match = re.search(r"FINAL_ANSWER:\s*([\d.]+)", cot_text)
if match:
    print("Parsed final answer:", match.group(1))
else:
    print("Could not parse FINAL_ANSWER. Full text:\n", cot_text)

---

## 10. Common CoT Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| `max_tokens` too low | Truncated reasoning, wrong answer | Increase limit |
| No final answer format | Hard to parse | `FINAL_ANSWER:` or JSON |
| CoT on trivial tasks | Wasted tokens | Use direct prompts |
| Trusting reasoning blindly | Fluent but wrong steps | Verify or use voting |
| High temperature on math | Inconsistent steps | Use 0–0.3 |
| Hiding steps from users | UX leak if you show raw output | Summarize or use reasoning models with separate channels |

---

## 11. Summary

**Chain-of-thought** prompting elicits intermediate steps before the final answer, improving many math, logic, and multi-step tasks. **Zero-shot CoT** uses short triggers; **few-shot CoT** demonstrates full reasoning traces.

CoT costs more tokens and latency — skip it for simple classification. Raise `max_tokens`, use low temperature for deterministic reasoning, and parse a clear **final answer** line.

**Self-consistency** samples multiple CoT paths and votes on the answer for extra reliability at N× cost. **Reasoning models** complement prompted CoT for the hardest problems.

Demonstrations compared direct vs zero-shot CoT on word problems and few-shot CoT with worked examples. Next: **05. Role Prompting and System Instructions** — shaping global behavior with system messages.